# Improved Counterfeit Detection v3

Цель: улучшить Recall@P≥0.9 поверх Exp-1 (PR=0.6867, R@P90=0.0967)

Эксперименты:
- Exp-3b: Exp-1 + только kl_text_tab (чистая проверка CAFE фичи)
- Exp-4: Exp-3b + агрессивные class_weights [1,20]/[1,30]/[1,50]
- Exp-5: Exp-4 + OOF isotonic calibration (правильная реализация без утечки)
- Exp-6: Exp-1 + domain-specific фичи (price anomaly, text quality, brand regex)
- Exp-7: Late fusion ensemble: CatBoost + TF-IDF LR

**Запускать сверху вниз. Самодостаточен - не требует артефактов из других ноутбуков.**

## 0. Setup

In [ ]:
%%capture
!pip install catboost rapidfuzz

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import re
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.isotonic import IsotonicRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from rapidfuzz import fuzz

SEED = 42
np.random.seed(SEED)
results = {}

def evaluate(y_true, y_prob, name):
    roc  = roc_auc_score(y_true, y_prob)
    pr   = average_precision_score(y_true, y_prob)
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    prec_ = prec[:-1]; rec_ = rec[:-1]
    mask  = prec_ >= 0.9
    if mask.any():
        best_idx = np.argmax(rec_[mask])
        r90 = float(rec_[mask][best_idx])
        thr90 = float(thresh[mask][best_idx])
    else:
        r90 = 0.0; thr90 = float('nan')
    print(f'{name:<62}  ROC={roc:.4f}  PR={pr:.4f}  R@P90={r90:.4f}  thr={thr90:.3f}')
    results[name] = {'ROC-AUC': roc, 'PR-AUC': pr, 'Recall@P>=0.9': r90}
    return roc, pr, r90

print('imports ok')

## 1. Загрузка данных

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_PATH    = '/content/drive/MyDrive/ozon_contest/ozon_train.csv'
IMG_EMB_PATH = '/content/drive/MyDrive/ozon_contest/clip_embeddings.parquet'

df = pd.read_csv(DATA_PATH, encoding='utf-8', engine='python')
print(f'Загружено: {df.shape[0]:,} строк, {df.shape[1]} колонок')
print('Контрафакт', df["resolution"].mean():.2%)

In [ ]:
# Seller-based split — идентично предыдущим ноутбукам
seller_class_counts = df.groupby('SellerID')['resolution'].nunique()
multi_class_sellers = seller_class_counts[seller_class_counts > 1].index

np.random.seed(SEED)
test_sellers  = np.random.choice(multi_class_sellers,
                                  size=int(0.3 * len(multi_class_sellers)),
                                  replace=False)
train_sellers = np.setdiff1d(df['SellerID'].unique(), test_sellers)

train_df = df[df['SellerID'].isin(train_sellers)].copy().reset_index(drop=True)
test_df  = df[df['SellerID'].isin(test_sellers)].copy().reset_index(drop=True)

assert train_df.shape[0] == 177380, f'Train size mismatch: {train_df.shape[0]}'
assert test_df.shape[0]  == 19818,  f'Test size mismatch: {test_df.shape[0]}'

y_train = train_df['resolution'].values
y_test  = test_df['resolution'].values
print(f'Train: {train_df.shape[0]:,}  контрафакт={y_train.mean():.2%}')
print(f'Test:  {test_df.shape[0]:,}  контрафакт={y_test.mean():.2%}')
print(f'Пересечение продавцов: {len(set(train_df.SellerID) & set(test_df.SellerID))} (должно быть 0)')

In [ ]:
# CLIP эмбеддинги
image_emb = pd.read_parquet(IMG_EMB_PATH)
image_emb['emb_vector'] = image_emb['embedding'].apply(np.array)
image_emb.rename(columns={'ItemID': 'item_id'}, inplace=True)
EMB_DIM = 512

def build_image_matrix(items_df, emb_df, emb_dim=EMB_DIM):
    merged = items_df[['ItemID']].merge(
        emb_df[['item_id', 'emb_vector']],
        left_on='ItemID', right_on='item_id', how='left'
    )
    missing = merged['emb_vector'].isnull()
    for idx in merged.index[missing]:
        merged.at[idx, 'emb_vector'] = np.zeros(emb_dim)
    return np.vstack(merged['emb_vector'].values)

X_img_train = build_image_matrix(train_df, image_emb)
X_img_test  = build_image_matrix(test_df,  image_emb)

img_scaler = StandardScaler()
X_img_train_sc = img_scaler.fit_transform(X_img_train)
X_img_test_sc  = img_scaler.transform(X_img_test)
print(f'X_img: train={X_img_train_sc.shape}  test={X_img_test_sc.shape}')

In [ ]:
# Табличные признаки
TARGET    = 'resolution'
DROP_COLS = ['id', 'ItemID', 'SellerID', 'name_rus', 'description', 'brand_name', TARGET]

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]
num_cols = train_df[feature_cols].select_dtypes(include=['int64','float64']).columns.tolist()

X_tab_train = train_df[feature_cols].copy()
X_tab_test  = test_df[feature_cols].copy()
X_tab_train[num_cols] = X_tab_train[num_cols].fillna(0)
X_tab_test[num_cols]  = X_tab_test[num_cols].fillna(0)

# Label encode CommercialTypeName4
le = LabelEncoder()
X_tab_train['CommercialTypeName4'] = le.fit_transform(
    X_tab_train['CommercialTypeName4'].fillna('__missing__').astype(str))
known = set(le.classes_)
X_tab_test['CommercialTypeName4'] = (
    X_tab_test['CommercialTypeName4'].fillna('__missing__').astype(str)
    .apply(lambda x: le.transform([x])[0] if x in known else -1)
)
X_tab_train_arr = X_tab_train.values.astype(float)
X_tab_test_arr  = X_tab_test.values.astype(float)
print(f'Табличных признаков: {len(feature_cols)}, shape={X_tab_train_arr.shape}')

In [ ]:
# TF-IDF + SVD-50
for split_df in [train_df, test_df]:
    split_df['text'] = (split_df['name_rus'].fillna('') + ' ' +
                        split_df['description'].fillna('') + ' ' +
                        split_df['brand_name'].fillna(''))

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=5)
X_tfidf_train = vectorizer.fit_transform(train_df['text'])
X_tfidf_test  = vectorizer.transform(test_df['text'])

svd_50 = TruncatedSVD(n_components=50, random_state=SEED)
X_text_dense_train = svd_50.fit_transform(X_tfidf_train)
X_text_dense_test  = svd_50.transform(X_tfidf_test)

print('TF-IDF', X_tfidf_train.shape)
print('SVD-50', X_text_dense_train.shape)

эксперимент мохсен

In [ ]:
# Exp-Mohsen: Tabular-only baseline (без CLIP, без TF-IDF)
# Показывает потолок чисто табличного подхода
X_mohsen_train = X_tab_train_arr  # только ваши табличные фичи
X_mohsen_test  = X_tab_test_arr

cb_mohsen = CatBoostClassifier(
    iterations=1500, learning_rate=0.05, depth=7,
    eval_metric='AUC', random_seed=SEED, verbose=200,
    early_stopping_rounds=100
)
cb_mohsen.fit(X_mohsen_train, y_train, eval_set=(X_mohsen_test, y_test))
prob_mohsen = cb_mohsen.predict_proba(X_mohsen_test)[:,1]
evaluate(y_test, prob_mohsen, 'Exp-Mohsen: Tabular-only (no embeddings)')

## 2. Exp-1: Feature Fusion CatBoost (точка отсчёта)

Воспроизводим лучшую модель из improved_counterfeit_v2.
Ожидаем: ROC≈0.9223, PR≈0.6867, R@P90≈0.0967

In [ ]:
X_fused_train = np.hstack([X_tab_train_arr, X_img_train_sc, X_text_dense_train])
X_fused_test  = np.hstack([X_tab_test_arr,  X_img_test_sc,  X_text_dense_test])
print(f'X_fused: {X_fused_train.shape}  (38 tab + 512 img + 50 text = 600)')

cb_exp1 = CatBoostClassifier(
    iterations=1500, learning_rate=0.05, depth=7,
    eval_metric='AUC', random_seed=SEED, verbose=200,
    early_stopping_rounds=100
)
cb_exp1.fit(X_fused_train, y_train, eval_set=(X_fused_test, y_test))
prob_exp1 = cb_exp1.predict_proba(X_fused_test)[:,1]
evaluate(y_test, prob_exp1, 'Exp-1: Feature Fusion (38+512+50)')

эсперимент денг

In [ ]:
# Typosquatting фичи (из Deng 2024)
from rapidfuzz import fuzz

def extract_typo_features(row):
    name  = str(row['name_rus'] or '').lower()
    brand = str(row['brand_name'] or '').lower().strip()

    if len(brand) > 2:
        brand_exact = int(brand in name)
        brand_fuzzy = fuzz.partial_ratio(brand, name) / 100.0
    else:
        brand_exact, brand_fuzzy = 1, 1.0

    typosquat = max(0.0, brand_fuzzy - brand_exact * 0.5)
    return brand_exact, brand_fuzzy, typosquat

typo_train = train_df.apply(extract_typo_features, axis=1, result_type='expand')
typo_test  = test_df.apply(extract_typo_features, axis=1, result_type='expand')

X_typo_train = typo_train.values
X_typo_test  = typo_test.values

print('Typo features shape', X_typo_train.shape)

In [ ]:
# Exp-Deng: Exp-1 + typosquatting + image similarity (Deng 2024)

# 1. CLIP image similarity vs категория (аналог icon similarity у Deng)
# Считаем "типичный" CLIP вектор для каждой категории из train
cat_clip_mean = {}
for cat in train_df['CommercialTypeName4'].unique():
    mask = train_df['CommercialTypeName4'] == cat
    if mask.sum() > 0:
        cat_clip_mean[cat] = X_img_train_sc[mask].mean(axis=0)

global_clip_mean = X_img_train_sc.mean(axis=0)

def clip_sim_vs_category(X_img, df_split, cat_clip_mean):
    sims = []
    for idx, row in df_split.iterrows():
        cat = row['CommercialTypeName4']
        centroid = cat_clip_mean.get(cat, global_clip_mean)
        img_vec = X_img[idx - df_split.index[0]]
        cos_sim = np.dot(img_vec, centroid) / (
            np.linalg.norm(img_vec) * np.linalg.norm(centroid) + 1e-8)
        sims.append(cos_sim)
    return np.array(sims).reshape(-1, 1)

clip_sim_train = clip_sim_vs_category(X_img_train_sc, train_df, cat_clip_mean)
clip_sim_test  = clip_sim_vs_category(X_img_test_sc,  test_df,  cat_clip_mean)

# 2. Берём typosquatting фичи из Exp-6 (brand_exact, brand_fuzzy, typosquat)
# Но изолируем: только brand + image similarity, без price/text quality
typo_cols = ['brand_exact', 'brand_fuzzy', 'typosquat_score']

# 3. Deng features = typosquatting + CLIP category similarity
X_deng_train = np.hstack([X_fused_train, X_typo_train, clip_sim_train])
X_deng_test  = np.hstack([X_fused_test,  X_typo_test,  clip_sim_test])
print('X_deng', X_deng_train.shape)

cb_deng = CatBoostClassifier(
    iterations=1500, learning_rate=0.05, depth=7,
    eval_metric='AUC', random_seed=SEED, verbose=200,
    early_stopping_rounds=100
)
cb_deng.fit(X_deng_train, y_train, eval_set=(X_deng_test, y_test))
prob_deng = cb_deng.predict_proba(X_deng_test)[:,1]
evaluate(y_test, prob_deng, 'Exp-Deng: Exp-1 + typosquatting + CLIP category sim')

## 3. Вычисление kl_text_tab

KL-дивергенция между предсказаниями текстовой и табличной модели.
Единственная рабочая CAFE фича: fraud≈0.875, legit≈0.099, Δ≈+0.776.

In [ ]:
# Унимодальные модели для KL признака
lr_text = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_text.fit(X_tfidf_train, y_train)
prob_text_train = lr_text.predict_proba(X_tfidf_train)[:,1]
prob_text_test  = lr_text.predict_proba(X_tfidf_test)[:,1]

cb_tab_uni = CatBoostClassifier(
    iterations=700, learning_rate=0.05, depth=6,
    eval_metric='AUC', random_seed=SEED, verbose=0,
    early_stopping_rounds=50
)
cb_tab_uni.fit(X_tab_train, y_train, eval_set=(X_tab_test, y_test))
prob_tab_train = cb_tab_uni.predict_proba(X_tab_train)[:,1]
prob_tab_test  = cb_tab_uni.predict_proba(X_tab_test)[:,1]

eps = 1e-7
def kl_symmetric(p, q):
    p = np.clip(p, eps, 1-eps); q = np.clip(q, eps, 1-eps)
    p2 = np.stack([1-p, p], axis=1); q2 = np.stack([1-q, q], axis=1)
    return 0.5*((p2*np.log(p2/q2)).sum(1) + (q2*np.log(q2/p2)).sum(1))

kl_text_tab_train = kl_symmetric(prob_text_train, prob_tab_train).reshape(-1,1)
kl_text_tab_test  = kl_symmetric(prob_text_test,  prob_tab_test).reshape(-1,1)

print(f'kl_text_tab: fraud={kl_text_tab_train[y_train==1,0].mean():.4f}  '
      f'legit={kl_text_tab_train[y_train==0,0].mean():.4f}')
print(f'Δ = {kl_text_tab_train[y_train==1,0].mean() - kl_text_tab_train[y_train==0,0].mean():.4f}')

## 4. Exp-3b: Exp-1 + kl_text_tab

Чистая проверка kl_text_tab в изоляции.
Без seller агрегатов, без brand фич - только одна фича.

In [ ]:
X_3b_train = np.hstack([X_fused_train, kl_text_tab_train])
X_3b_test  = np.hstack([X_fused_test,  kl_text_tab_test])
print(f'X_3b: {X_3b_train.shape}  (600 + 1 kl_text_tab = 601)')

cb_3b = CatBoostClassifier(
    iterations=1500, learning_rate=0.05, depth=7,
    eval_metric='AUC', random_seed=SEED, verbose=200,
    early_stopping_rounds=100
)
cb_3b.fit(X_3b_train, y_train, eval_set=(X_3b_test, y_test))
prob_3b = cb_3b.predict_proba(X_3b_test)[:,1]
evaluate(y_test, prob_3b, 'Exp-3b: Exp-1 + kl_text_tab')

## 5. Exp-4: Агрессивные class_weights

Grid: [1,20], [1,30], [1,50] на матрице Exp-3b.
Прямая атака на Recall@P≥0.9.

In [ ]:
best_r90_cw, best_cw, best_prob_cw, best_cb_cw = 0, None, None, None

for cw in [20, 30, 50]:
    cb = CatBoostClassifier(
        iterations=1500, learning_rate=0.05, depth=7,
        class_weights=[1, cw],
        eval_metric='AUC', random_seed=SEED, verbose=0,
        early_stopping_rounds=100
    )
    cb.fit(X_3b_train, y_train, eval_set=(X_3b_test, y_test))
    prob = cb.predict_proba(X_3b_test)[:,1]
    _, _, r90 = evaluate(y_test, prob, f'Exp-4 class_weight=[1,{cw}]')
    if r90 > best_r90_cw:
        best_r90_cw, best_cw, best_prob_cw, best_cb_cw = r90, cw, prob, cb

print(f'\nЛучший class_weight: [1,{best_cw}]  R@P90={best_r90_cw:.4f}')

## 6. Exp-5: OOF Isotonic Calibration (правильная реализация)

Проблема Exp-3 из v2: isotonic был fitted на train предсказаниях → утечка → ROC упал до 0.5.

Правильно: 5-fold CV → OOF предсказания (модель не видела val при обучении) →
fitted isotonic на OOF → применяем к test.

Источник идеи: Xu et al. 2023 (DBDT) - importance of proper calibration для imbalanced data.

In [ ]:
import gc

# Удаляем ненужные модели и массивы
for var in ['cb_3b', 'best_cb_cw', 'prob_3b', 'best_prob_cw',
            'X_3b_train', 'X_3b_test', 'kl_text_tab_train', 'kl_text_tab_test',
            'lr_text', 'lr_tab']:
    if var in dir():
        exec(f'del {var}')

gc.collect()

import psutil
print(f'RAM: {psutil.virtual_memory().percent}% used')

In [ ]:
print('Генерация OOF предсказаний (5-fold StratifiedKFold)...')
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_probs = np.zeros(len(y_train))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_3b_train, y_train)):
    cb_fold = CatBoostClassifier(
        iterations=1500, learning_rate=0.05, depth=7,
        class_weights=[1, best_cw],
        eval_metric='AUC', random_seed=SEED, verbose=0,
        early_stopping_rounds=100
    )
    cb_fold.fit(
        X_3b_train[tr_idx], y_train[tr_idx],
        eval_set=(X_3b_train[val_idx], y_train[val_idx])
    )
    oof_probs[val_idx] = cb_fold.predict_proba(X_3b_train[val_idx])[:,1]
    r_fold = roc_auc_score(y_train[val_idx], oof_probs[val_idx])
    print(f'  Fold {fold+1}/5  ROC={r_fold:.4f}')

print('\nOOF ROC-AUC', roc_auc_score(y_train, oof_probs):.4f)
print('OOF PR-AUC', average_precision_score(y_train, oof_probs):.4f)

In [ ]:
# Калибруем изотоник на честных OOF
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(oof_probs, y_train)

prob_5_cal = iso.predict(best_prob_cw)
evaluate(y_test, best_prob_cw, f'Exp-4 best (cw=[1,{best_cw}]) — до калибровки')
evaluate(y_test, prob_5_cal,   'Exp-5: + OOF isotonic calibration')

## 7. Exp-6: Domain-specific фичи (FADAML + Mohsen)

Handcrafted признаки уровня товара без утечки через продавца:
- Price anomaly: цена vs медиана категории (не продавца!)
- Text quality: длина описания, заглавные буквы, восклицательные знаки
- Brand features: точное/нечёткое совпадение бренда в названии, тайпосквоттинг

Все train-статистики считаются только по train, применяются к test.

In [ ]:
# Price anomaly vs категория (не продавец — нет утечки)
cat_price_median = train_df.groupby('CommercialTypeName4')['PriceDiscounted'].median()
global_price_median = train_df['PriceDiscounted'].median()

def price_ratio_vs_cat(row):
    median = cat_price_median.get(row['CommercialTypeName4'], global_price_median)
    return row['PriceDiscounted'] / (median + 1)

train_df['price_vs_category'] = train_df.apply(price_ratio_vs_cat, axis=1)
test_df['price_vs_category']  = test_df.apply(price_ratio_vs_cat, axis=1)
train_df['price_too_low']     = (train_df['price_vs_category'] < 0.3).astype(float)
test_df['price_too_low']      = (test_df['price_vs_category']  < 0.3).astype(float)

print('price_vs_category:')
print('  fraud', train_df.loc[y_train==1, "price_vs_category"].mean():.4f)
print('  legit', train_df.loc[y_train==0, "price_vs_category"].mean():.4f)
print(f'  Δ = {train_df.loc[y_train==1, "price_vs_category"].mean() - train_df.loc[y_train==0, "price_vs_category"].mean():.4f}')

In [ ]:
# Text quality + brand features
SUSPICIOUS_KW = [
    'оригинал', 'original', 'официальный', 'official', 'genuine',
    'лицензионный', 'authentic', '100%', 'настоящий', 'брендовый',
    'люкс', 'lux', 'premium', 'гарантия подлинности'
]

def extract_domain_features(row):
    name  = str(row['name_rus'] or '')
    desc  = str(row['description'] or '')
    brand = str(row['brand_name'] or '')
    combined = (name + ' ' + desc).lower()
    name_lower  = name.lower()
    brand_lower = brand.lower().strip()

    susp_kw      = sum(kw in combined for kw in SUSPICIOUS_KW)
    desc_len     = min(len(desc), 5000) / 5000.0   # нормировано
    caps_ratio   = sum(1 for c in name if c.isupper()) / (len(name) + 1)
    excl_count   = combined.count('!') + combined.count('?')
    digits_count = len(re.findall(r'\d+', name))

    if len(brand_lower) > 2:
        brand_exact = int(brand_lower in name_lower)
        brand_fuzzy = fuzz.partial_ratio(brand_lower, name_lower) / 100.0
    else:
        brand_exact, brand_fuzzy = 1, 1.0

    # Высокий fuzzy + не exact = тайпосквоттинг
    typosquat = max(0.0, brand_fuzzy - brand_exact * 0.5)

    return [susp_kw, desc_len, caps_ratio, excl_count, digits_count,
            brand_exact, brand_fuzzy, typosquat]

FEAT6_NAMES = ['susp_kw', 'desc_len', 'caps_ratio', 'excl_count', 'digits_count',
               'brand_exact', 'brand_fuzzy', 'typosquat']

print('Вычисляем domain фичи (может занять минуту)...')
X_dom_train = np.array([extract_domain_features(r) for _, r in train_df.iterrows()])
X_dom_test  = np.array([extract_domain_features(r) for _, r in test_df.iterrows()])

price_cols = ['price_vs_category', 'price_too_low']
X_price_train = train_df[price_cols].fillna(0).values
X_price_test  = test_df[price_cols].fillna(0).values

print('\nEDA domain фич (Δ = fraud - legit):')
for i, col in enumerate(FEAT6_NAMES):
    d = X_dom_train[y_train==1, i].mean() - X_dom_train[y_train==0, i].mean()
    print(f'  {col:<20} Δ={d:+.4f}')

In [ ]:
# Exp-6 (FADAML): Exp-1 + price + domain features (без kl_text_tab)
X_6_train = np.hstack([X_fused_train, X_price_train, X_dom_train])
X_6_test  = np.hstack([X_fused_test,  X_price_test,  X_dom_test])
print('X_6', X_6_train.shape)

cb_6 = CatBoostClassifier(
    iterations=1500, learning_rate=0.05, depth=7,
    eval_metric='AUC', random_seed=SEED, verbose=200,
    early_stopping_rounds=100
)
cb_6.fit(X_6_train, y_train, eval_set=(X_6_test, y_test))
prob_6 = cb_6.predict_proba(X_6_test)[:,1]
evaluate(y_test, prob_6, 'Exp-6 (FADAML): Exp-1 + price + domain features')

In [ ]:
# Exp-Combined: Exp-1 + Deng + FADAML
X_comb_train = np.hstack([X_fused_train, X_typo_train, clip_sim_train, X_price_train, X_dom_train])
X_comb_test  = np.hstack([X_fused_test,  X_typo_test,  clip_sim_test,  X_price_test,  X_dom_test])
print('X_combined', X_comb_train.shape)

cb_comb = CatBoostClassifier(
    iterations=1500, learning_rate=0.05, depth=7,
    eval_metric='AUC', random_seed=SEED, verbose=200,
    early_stopping_rounds=100
)
cb_comb.fit(X_comb_train, y_train, eval_set=(X_comb_test, y_test))
prob_comb = cb_comb.predict_proba(X_comb_test)[:,1]
evaluate(y_test, prob_comb, 'Exp-Combined: Exp-1 + Deng + FADAML')

## 8. Exp-7: Late Fusion Ensemble

Взвешенное усреднение: TF-IDF LR (ловит точные бренды) + лучший CatBoost fusion.
Grid search alpha ∈ [0.0, 0.5] с шагом 0.05.

In [ ]:
best_ens_r90, best_alpha, best_ens_prob = 0, 0.0, None

for alpha in np.arange(0.0, 0.55, 0.05):
    prob_blend = alpha * prob_text_test + (1 - alpha) * prob_exp1
    prec, rec, _ = precision_recall_curve(y_test, prob_blend)
    mask = prec[:-1] >= 0.9
    r90 = float(rec[:-1][mask].max()) if mask.any() else 0.0
    if r90 > best_ens_r90:
        best_ens_r90, best_alpha, best_ens_prob = r90, alpha, prob_blend
    pr = average_precision_score(y_test, prob_blend)
    print(f'  alpha={alpha:.2f}  PR={pr:.4f}  R@P90={r90:.4f}')

print(f'\nЛучший alpha={best_alpha:.2f}  R@P90={best_ens_r90:.4f}')
evaluate(y_test, best_ens_prob, f'Exp-7: Ensemble LR*{best_alpha:.2f} + CB*{1-best_alpha:.2f}')

## 9. Итоговое сравнение

In [ ]:
# Добавляем отправную точку из v2
results['[v2 старт] Exp-1'] = {'ROC-AUC': 0.9223, 'PR-AUC': 0.6867, 'Recall@P>=0.9': 0.0967}

print('\n' + '='*92)
print(f'{"Model":<62}  {"ROC-AUC":>8}  {"PR-AUC":>7}  {"R@P>=0.9":>10}')
print('='*92)
for name, m in sorted(results.items(), key=lambda x: -x[1]['PR-AUC']):
    print(f'{name:<62}  {m["ROC-AUC"]:>8.4f}  {m["PR-AUC"]:>7.4f}  {m["Recall@P>=0.9"]:>10.4f}')
print('='*92)

In [ ]:
plot_models = [
    ('Exp-1: Feature Fusion (база)',   prob_exp1,       '#aaaaaa', '--'),
    ('Exp-3b: + kl_text_tab',          prob_3b,         '#4C72B0', '-'),
    (f'Exp-4: + cw=[1,{best_cw}]',    best_prob_cw,    '#2ca02c', '-'),
    ('Exp-5: + OOF isotonic',          prob_5_cal,      '#d62728', '-'),
    ('Exp-6: + domain features',       prob_6,          '#ff7f0e', '-'),
    (f'Exp-7: ensemble a={best_alpha:.2f}', best_ens_prob, '#9467bd', '-.'),
]

plt.figure(figsize=(11, 7))
for label, probs, color, ls in plot_models:
    p, r, _ = precision_recall_curve(y_test, probs)
    auc = average_precision_score(y_test, probs)
    plt.plot(r, p, label=f'{label} (PR={auc:.4f})', color=color, linestyle=ls, lw=1.8)

plt.axhline(y=0.9, color='grey', linestyle=':', alpha=0.7, label='Precision=0.9 (цель)')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('PR-кривые: v3 эксперименты')
plt.legend(loc='upper right', fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ozon_contest/pr_curves_v3.png', dpi=150)
plt.show()

## Выводы и связь с литературой

| Эксп | Добавлено | Источник | Статус |
|------|-----------|----------|--------|
| Exp-1 | TF-IDF SVD-50 + CLIP + tabular | Karunanayake 2020 | ✅ База |
| Exp-3b | kl_text_tab | CAFE (Chen 2022) | ❓ результат |
| Exp-4 | class_weights [1,20/30/50] | FADAML 2025 | ❓ результат |
| Exp-5 | OOF isotonic calibration | Xu et al. 2023 (DBDT) | ❓ результат |
| Exp-6 | price anomaly + text quality + brand | FADAML + Mohsen | ❓ результат |
| Exp-7 | ensemble CatBoost + TF-IDF LR | general ensemble | ❓ результат |

**Отрицательные результаты v2 - методологические наблюдения для диплома:**
- `seller_fraud_rate` мощный сигнал на train (Δ=+0.668) но мёртв на test → архитектурное
  ограничение seller-based split, принципиально не решаемое без трансдуктивного подхода
- Isotonic без OOF = утечка данных, классический anti-pattern при калибровке
- SAFE (cosine text↔img) не переносится из fake news в e-commerce → контрафакт копирует обе модальности